<a href="https://colab.research.google.com/github/collinskiplangatrotich/plp-da-a2-rainfall-advisory/blob/main/rainfall_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Regional Rainfall Analysis & Planting Advisory System

Course: Data Analytics with Python (PLP Academy)

Assignment: Week 2 - Python Data Processing with NumPy

Author: Collins Kiplangat Rotich

Task 1: Dataset Initialization & Machakos Data Cleaning


In this section, we define the dataset for four clean counties and write a function using explicit try/except ValueError to parse and clean machakos_raw. Missing/invalid values are imputed with the rounded mean of valid Machakos entries, and imputed months are explicitly printed.

In [23]:

# Provided Dataset
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
months = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

def clean_machakos_data(raw_data, month_names):
    """
    Cleans Machakos raw text rainfall readings.
    Converts valid values to integers using try/except ValueError.
    Imputes invalid entries with the rounded mean of valid values.
    """
    valid_values = []
    # First pass: Extract valid integer readings
    for entry in raw_data:
        try:
            val = int(str(entry).strip())
            valid_values.append(val)
        except ValueError:
            pass

    # Calculate rounded average of valid figures
    avg_val = round(sum(valid_values) / len(valid_values))

    cleaned = []
    # Second pass: Replace invalid entries with imputed mean
    for idx, entry in enumerate(raw_data):
        try:
            val = int(str(entry).strip())
            cleaned.append(val)
        except ValueError:
            print(f"Imputed month at index {idx} ({month_names[idx]}) with average value: {avg_val} mm")
            cleaned.append(avg_val)

    return cleaned

# Execute Machakos Data Cleaning
machakos_clean = clean_machakos_data(machakos_raw, months)
print("Cleaned Machakos Rainfall Data (mm):", machakos_clean)

Imputed month at index 3 (April) with average value: 68 mm
Imputed month at index 8 (September) with average value: 68 mm
Cleaned Machakos Rainfall Data (mm): [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


Task 2: Combine Counties into a 5 × 12 NumPy Array
​


We merge all five cleaned county lists into a unified 2D NumPy array (5 x 12) and verify its array dimensions (shape) and data type (dtype).

In [24]:
import numpy as np

# Combine all 5 county rows
all_counties_list = clean_rows + [machakos_clean]
rainfall = np.array(all_counties_list, dtype=np.int64)

print("Rainfall Array Shape:", rainfall.shape)
print("Rainfall Array Data Type:", rainfall.dtype)
print("Rainfall Matrix (5x12):\n", rainfall)

Rainfall Array Shape: (5, 12)
Rainfall Array Data Type: int64
Rainfall Matrix (5x12):
 [[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]


Task 3: Vectorized Summary Calculations (Annual Totals & Monthly Averages)
​


Using vectorized NumPy axis aggregation functions (np.sum across rows and np.mean across columns), we compute the total annual rainfall per county and regional average rainfall per month without Python loops.

In [25]:

# Vectorized Annual Totals per County (Sum across columns axis=1)
annual_totals = np.sum(rainfall, axis=1)

# Vectorized Monthly Averages across all Counties (Mean across rows axis=0)
monthly_averages = np.mean(rainfall, axis=0)

print("--- Annual Rainfall Total Per County ---")
for county, total in zip(counties, annual_totals):
    print(f"{county}: {total:,} mm")

print("\n--- Regional Monthly Averages Across All Counties ---")
for month_name, avg_val in zip(months, monthly_averages):
    print(f"{month_name}: {avg_val:.2f} mm")

--- Annual Rainfall Total Per County ---
Nairobi: 1,045 mm
Nakuru: 916 mm
Kisumu: 1,371 mm
Mombasa: 1,075 mm
Machakos: 811 mm

--- Regional Monthly Averages Across All Counties ---
January: 47.00 mm
February: 47.60 mm
March: 91.00 mm
April: 164.40 mm
May: 171.20 mm
June: 67.60 mm
July: 56.60 mm
August: 58.20 mm
September: 64.20 mm
October: 67.80 mm
November: 127.40 mm
December: 80.60 mm


Task 4: Wettest Month Identification using np.argmax
​


We use np.argmax(rainfall, axis=1) to locate the column index of peak rainfall for each county and pair the indices with month names using zip().

In [26]:

# Find indices of wettest months per county
wettest_indices = np.argmax(rainfall, axis=1)
wettest_months = [months[idx] for idx in wettest_indices]

print("--- Wettest Month Per County ---")
for county, month_name, idx in zip(counties, wettest_months, range(len(counties))):
    peak_val = rainfall[idx, wettest_indices[idx]]
    print(f"{county}: {month_name} ({peak_val} mm)")

--- Wettest Month Per County ---
Nairobi: April (219 mm)
Nakuru: April (145 mm)
Kisumu: April (240 mm)
Mombasa: May (240 mm)
Machakos: November (182 mm)


Task 5: Dry-Month Counting using Boolean Masking
​


We count the number of dry months (< 50 mm) for each county using a vectorized boolean comparison (rainfall < 50) and row-wise summation np.sum(..., axis=1).

In [27]:

# Vectorized Boolean Mask for Dry Months (< 50 mm)
dry_mask = rainfall < 50
dry_month_counts = np.sum(dry_mask, axis=1)

print("--- Dry Months Count Per County (< 50 mm) ---")
for county, count in zip(counties, dry_month_counts):
    print(f"{county}: {count} dry months (< 50 mm)")

--- Dry Months Count Per County (< 50 mm) ---
Nairobi: 4 dry months (< 50 mm)
Nakuru: 3 dry months (< 50 mm)
Kisumu: 0 dry months (< 50 mm)
Mombasa: 2 dry months (< 50 mm)
Machakos: 6 dry months (< 50 mm)


Task 6: Optimal 2-Month Planting Window Identification
​


Using array shifting rainfall[:, :-1] + rainfall[:, 1:], we calculate the total combined rainfall for all consecutive 2-month periods and identify the highest volume window for each county.

In [28]:

# Vectorized Array Shifting for 2-Month Rolling Totals
two_month_totals = rainfall[:, :-1] + rainfall[:, 1:]

# Identify index of peak 2-month window per county
window_indices = np.argmax(two_month_totals, axis=1)

planting_windows = []
print("--- Recommended 2-Month Planting Windows ---")
for i, (county, start_idx) in enumerate(zip(counties, window_indices)):
    w_name = f"{months[start_idx]}-{months[start_idx+1]}"
    w_val = two_month_totals[i, start_idx]
    planting_windows.append(w_name)
    print(f"{county}: {w_name} ({w_val} mm)")

--- Recommended 2-Month Planting Windows ---
Nairobi: April-May (395 mm)
Nakuru: April-May (265 mm)
Kisumu: April-May (420 mm)
Mombasa: April-May (390 mm)
Machakos: November-December (278 mm)


In [29]:

import csv

# Export results to rainfall_summary.csv
csv_filename = "rainfall_summary.csv"
headers = ["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"]

rows_to_write = []
for i in range(len(counties)):
    rows_to_write.append({
        "county": counties[i],
        "annual_total_mm": int(annual_totals[i]),
        "wettest_month": wettest_months[i],
        "dry_months": int(dry_month_counts[i]),
        "planting_window": planting_windows[i]
    })

with open(csv_filename, mode="w", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=headers)
    writer.writeheader()
    writer.writerows(rows_to_write)

print(f"File '{csv_filename}' generated successfully!\n")

# Verify written CSV content
with open(csv_filename, mode="r", encoding="utf-8") as f:
    print("--- Exported CSV File Contents ---")
    print(f.read())

File 'rainfall_summary.csv' generated successfully!

--- Exported CSV File Contents ---
county,annual_total_mm,wettest_month,dry_months,planting_window
Nairobi,1045,April,4,April-May
Nakuru,916,April,3,April-May
Kisumu,1371,April,0,April-May
Mombasa,1075,May,2,April-May
Machakos,811,November,6,November-December



## Data Quality Notes

The dataset received for regional rainfall analysis exhibits high reliability across Nairobi, Nakuru, Kisumu, and Mombasa. However, the raw station readings for Machakos County contained non-numeric text ('N/A') in April (index 3) and an empty string ('') in September (index 8).

To preserve dataset continuity and prevent bias in matrix operations, these two missing records were imputed using the rounded mean of the remaining 10 valid Machakos entries (68 mm). While this mathematical imputation prevents total record loss, it introduces artificial smoothing into the April long rains and September transition periods.

Future station telemetry should implement automated digital logging to eliminate manual data-entry errors.

## Planting Advisory

Based on the 5-county regional rainfall analysis, the farmer-support NGO should issue county-specific SMS advisories to maximize crop yields and manage drought risk. Kisumu, Nairobi, Nakuru, and Mombasa should schedule primary crop planting during the April–May window, which captures their highest cumulative rainfall (peaking at 420 mm in Kisumu and 390 mm in Mombasa).

Special management is required for Machakos, which peaks later during November–December (278 mm) and experiences 6 dry months (< 50 mm) per year; farmers in Machakos should implement rainwater harvesting, mulching, and short-cycle drought-tolerant seed varieties. Nakuru also requires careful monitoring due to 3 dry months annually.

Data Quality Caveat: Advice for Machakos County relies on imputed data for April and September (68 mm each) due to missing raw station logs; actual local rainfall during these two months may vary from station estimates, so farmers should cross-reference local soil moisture prior to heavy seed investment.